### Một số đạo hàm cục bộ trong quá trình backpropagation

Các công thức dưới đây được suy ra bằng chain rule trên đúng các phép toán dùng trong mô hình. Đây là đạo hàm cục bộ của một vài bước chính, không phải toàn bộ đạo hàm từ loss qua mọi bước của HOG.

#### Gate của orientation bin

Giả sử vote mềm của bin $k$ là:

$$
V_k = m\,a_k\,g_k
$$

Trong đó $m$ là gradient magnitude, $a_k$ là mức gán mềm vào bin và $g_k$ là gate. Khi xem $m$ và $a_k$ là đầu vào của bước này:

$$
\frac{\partial L}{\partial g_k}
=
\frac{\partial L}{\partial V_k}\,m\,a_k
$$

Trong code, optimizer cập nhật `gate_logit` $q_k$ chứ không cập nhật trực tiếp $g_k$. Do:

$$
g_k = \sigma(q_k)
$$

nên gradient theo tham số thật còn đi qua sigmoid:

$$
\frac{\partial L}{\partial q_k}
=
\frac{\partial L}{\partial g_k}\,g_k(1-g_k)
$$

Nếu bin đóng góp theo hướng làm giảm loss thì gate có thể tăng; nếu bin gây nhiễu thì gate có thể giảm. Các thành phần regularization của gate cũng đóng góp thêm vào gradient tổng.

#### Tâm bin hướng

Biểu diễn góc kép tại một pixel và vector tâm bin chuẩn hóa là:

$$
\mathbf{d}(\theta)
=
[\cos(2\theta),\sin(2\theta)]
$$

$$
\hat{\mathbf{u}}_k
=
[\cos(2\mu_k),\sin(2\mu_k)]
$$

Điểm số của bin $k$ có thể viết:

$$
s_k
=
\kappa_k\,\mathbf{d}(\theta)^T\hat{\mathbf{u}}_k
=
\kappa_k\cos(2\theta-2\mu_k)
$$

Softmax được tính trên điểm số của tất cả các bin:

$$
a_k
=
\operatorname{softmax}(\mathbf{s})_k
=
\frac{\exp(s_k)}{\sum_j \exp(s_j)}
$$

Nếu biểu diễn tâm bin bằng góc $\mu_k$, đạo hàm cục bộ là:

$$
\frac{\partial s_k}{\partial \mu_k}
=
2\kappa_k\sin(2\theta-2\mu_k)
$$

Code thực tế học vector thô $\mathbf{u}_k$ trong `bin_matrix`, sau đó chuẩn hóa:

$$
\hat{\mathbf{u}}_k
=
\frac{\mathbf{u}_k}{\lVert\mathbf{u}_k\rVert}
$$

Gradient vì vậy còn đi qua phép chuẩn hóa vector và softmax. Công thức theo $\mu_k$ là cách biểu diễn tương đương, dễ đọc hơn khi giải thích sự dịch chuyển của tâm bin.

#### Ngưỡng clip trong L2-Hys

Giả sử sau lần chuẩn hóa block thứ nhất có giá trị $x_i$. Sau clipping:

$$
z_i = \min(x_i,c)
$$

Đạo hàm cục bộ theo $c$ là:

$$
\frac{\partial z_i}{\partial c}
=
\begin{cases}
1, & x_i>c,\\
0, & x_i<c.
\end{cases}
$$

Đây là đạo hàm chính xác khi $x_i\ne c$. Tại $x_i=c$, hàm không khả vi theo nghĩa cổ điển và autograd sử dụng một quy ước subgradient tại điểm biên.

Trong code, ngưỡng clip được tham số hóa bởi `clip_raw` $r$:

$$
c = 0.05 + 0.45\sigma(r)
$$

Vì vậy, gradient theo tham số thật là:

$$
\frac{\partial L}{\partial r}
=
\frac{\partial L}{\partial c}
\,0.45\sigma(r)\bigl(1-\sigma(r)\bigr)
$$

Gradient $\partial L/\partial c$ ở đây đã bao gồm ảnh hưởng của lần chuẩn hóa L2 thứ hai và các bước tính loss phía sau.

#### Kernel Sobel học được

Trong PyTorch, `F.conv2d` thực hiện phép cross-correlation. Với một vị trí $p$, gradient ảnh theo trục $x$ có thể viết:

$$
G_x[p]
=
\sum_u I[p+u]K_x[u]
$$

Áp dụng chain rule cho từng phần tử của kernel:

$$
\frac{\partial L}{\partial K_x[u]}
=
\sum_p
I[p+u]
\frac{\partial L}{\partial G_x[p]}
$$

Đây là phép cộng gradient trên các image patch mà kernel đã đi qua. Kernel được khởi tạo từ Sobel nhưng được khai báo bằng `nn.Parameter`, nên gradient từ loss và các thành phần regularization có thể cập nhật kernel trong quá trình training.